Bill of Quantities - PV Toolestown I
=
This notebook counts the fixed-tilt structures and measures the trenches, ducts and cables drawn in the PV Toolestown I BOQ drawing. Lengths are summed **per layer** and written to `C:\Users\Usuario\Desktop\Toolestown I - BOQ.xlsx`. The layer name tells the item type and, through its suffix, the number of circuits (`NNxC`) or conduits (`NS`).

This is Toolestown I (PCS 01 to 05, layers `EMF_em_PCS_<nn>`). `BOQ - Toolestown (6-11).ipynb` is the same notebook for Toolestown II.

**Workflow overview:**
1. Convert the DWG to DXF and load all model space entities into `pv_projects.duckdb` (project `"PV Toolestown I"`).
2. Select the entities of the project as a DuckDB relation (`data`).
3. **Structures:** take every string block, group the blocks into east-west rows (same Y) and split each row where there is a gap in X. The number of groups is the number of structures (printed only, not exported).
4. **Cables:** select AC trenches, AC access ductbanks, DC cables, DC ducts and DC trenches by layer name, then sum `length` per layer and count the pieces.
5. Write every summary table to its own sheet and close the connection.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Toolestown I - BOQ.dwg` in the project's `01 - Layout` folder inside the synced Ecotechnee `Office - Documentos` folder (`dwg_folder`). The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** blocks and lines inside other blocks, xrefs or paper space aren't read.
- **String blocks:**
  - One block reference (`INSERT`) per string, named exactly `EMF422-2_string_tilt_15°` (exact match, case-sensitive, including the `°`).
  - Dynamic block references whose properties were changed get an anonymous name (`*U<n>`) and are **not** counted. `BOQ - Reask.ipynb` has to add `*U2` for this reason; check the block names in `data` for `*U` names (e.g. `SELECT name, COUNT(*) FROM data GROUP BY name`).
  - The structures run east-west: all the strings of one table row have the same insertion Y (rounded to the metre), and consecutive strings of a structure are at most 13 m apart in X (the pitch is ~11.7 m).
  - Two structures in the same row must be more than 13 m apart in X, otherwise they are counted as one.
- **Entity types:** trenches, ducts and cables must be `LINE` or `LWPOLYLINE` (only these get a `length`). Drawing units are metres.
- **Layer names** (case-sensitive):
  - AC trenches: `LIKE 'EMF_bt_vala_btca_%xC'`.
  - AC ducts in access crossings: `LIKE 'EMF_bt_access_ductbank_%xC'`.
  - DC cables: exactly `EMF_cabo_solar_negativo` and `EMF_cabo_solar_positivo`.
  - DC ducts: `LIKE 'EMF_bt_eletroduto_%S'`.
  - DC trenches: exactly `EMF_DC_vala` (a single layer, so there is no split by number of circuits).
  - In `LIKE`, `%` = any text and `_` also matches any single character.
- **Ducts and crossings are counted per entity:** each duct piece / crossing must be one single LINE or LWPOLYLINE.

Extracting the AutoCAD data and loading it into the DuckDB database
=
- Copies `PV Toolestown I - BOQ.dwg` from `dwg_folder` into `temp_folder`, so the ODA File Converter only converts that one file and not every DWG in the project folder.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) writes the DXF (ACAD2018) into `output_folder`, which is the same folder as the DWG.
- `load_to_duckdb` reads every model space entity with `ezdxf` and loads it into the `cad_entities` table under the project name `"PV Toolestown I"`. If the project already exists, its old rows are deleted first, so the table always holds the latest version of the drawing.
  - `LINE` / `LWPOLYLINE`: start/end point and length (for polylines, along all segments including bulge arcs).
  - `INSERT` (block reference): block name, insertion point (`X`, `Y`), rotation, color and the attribute values.
- At the end, the temporary DXF and the copied DWG are deleted. The DuckDB connection (`conn`) stays open for the rest of the notebook and is closed in the last cell.

In [2]:
from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\2 - Toolestown Solar\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout"
temp_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\2 - Toolestown Solar\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Toolestown I - BOQ.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Toolestown I - BOQ.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Toolestown I - BOQ.dwg")

conn = load_to_duckdb(dxf_file, "PV Toolestown I")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Toolestown I already exists
Removed existing data for project: PV Toolestown I
Successfully loaded project
Loaded 8059 entities for project: PV Toolestown I
Don't forget to close connection.
Data Loaded
DuckDB Loaded


CABLE AND STRUCTURE DATA
=

- `data` is a DuckDB relation with `name`, `layer`, insertion point (`X`, `Y`), start/end coordinates and `length` of project `"PV Toolestown I"`. The next cells query it by its Python variable name (DuckDB can query Python relations/DataFrames by name).
- `DESCRIBE cad_entities` prints the table schema.
- The layer count runs on `cad_entities` **without a project filter**, so it counts the layers of every project in the database, not only this one. Query `data` instead to see only this drawing.

In [3]:
data = conn.sql("SELECT name, layer, X, Y, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name='PV Toolestown I'")
print(data)
print(conn.execute("DESCRIBE cad_entities").fetchdf())

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM cad_entities
    GROUP BY layer
    ORDER BY count DESC
""").fetchdf()
print(result)

┌──────────────────────────┬─────────────────────────────┬───────────┬──────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│           name           │            layer            │     X     │    Y     │  start_x  │  start_y  │   end_x   │   end_y   │  length   │
│         varchar          │           varchar           │   float   │  float   │   float   │   float   │   float   │   float   │   float   │
├──────────────────────────┼─────────────────────────────┼───────────┼──────────┼───────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF422-2_string_tilt_15° │ EMF_em_PCS_04               │  692270.4 │ 735873.6 │      NULL │      NULL │      NULL │      NULL │      NULL │
│ EMF422-2_string_tilt_15° │ EMF_em_PCS_04               │  692282.1 │ 735873.6 │      NULL │      NULL │      NULL │      NULL │      NULL │
│ EMF422-2_string_tilt_15° │ EMF_em_PCS_04               │  692293.8 │ 735873.6 │      NULL │      NULL │      NULL │      NULL │      NULL │
│ EMF4

STRUCTURE DATA

The next two cells count the fixed structures from the string blocks.

Selecting the string blocks
=
Every block reference named exactly `EMF422-2_string_tilt_15°` (one per string). Only the layer (`EMF_em_PCS_<nn>`) and the insertion point are kept.

In [4]:
structures = conn.sql("SELECT layer, X, Y FROM data WHERE name = 'EMF422-2_string_tilt_15°'")
print(structures)

┌───────────────┬───────────┬───────────┐
│     layer     │     X     │     Y     │
│    varchar    │   float   │   float   │
├───────────────┼───────────┼───────────┤
│ EMF_em_PCS_04 │  692270.4 │  735873.6 │
│ EMF_em_PCS_04 │  692282.1 │  735873.6 │
│ EMF_em_PCS_04 │  692293.8 │  735873.6 │
│ EMF_em_PCS_04 │ 692305.56 │  735873.6 │
│ EMF_em_PCS_04 │ 692317.25 │  735873.6 │
│ EMF_em_PCS_04 │  692329.0 │  735873.6 │
│ EMF_em_PCS_04 │  692270.4 │  735863.9 │
│ EMF_em_PCS_04 │  692282.1 │  735863.9 │
│ EMF_em_PCS_04 │  692293.8 │  735863.9 │
│ EMF_em_PCS_04 │ 692305.56 │  735863.9 │
│       ·       │     ·     │      ·    │
│       ·       │     ·     │      ·    │
│       ·       │     ·     │      ·    │
│ EMF_em_PCS_01 │ 693035.75 │ 736280.56 │
│ EMF_em_PCS_01 │  693012.3 │ 736300.06 │
│ EMF_em_PCS_01 │  692927.8 │  736354.1 │
│ EMF_em_PCS_03 │ 692653.75 │  735854.6 │
│ EMF_em_PCS_03 │ 692661.25 │  735864.4 │
│ EMF_em_PCS_03 │  692649.5 │  735864.4 │
│ EMF_em_PCS_03 │  692657.3 │  735

Counting the structures
=
- `Line` = insertion Y rounded to the metre: the table row the string belongs to.
- Sort order: `Line` descending, then `X` ascending (row by row, north to south, west to east).
- A new structure starts when:
  1. the row changes (`Line` differs from the previous row), or
  2. the X gap to the previous string is larger than **13 m** (strings of one table are ~11.7 m apart).
- `Group` = cumulative sum of the break flags, so its maximum is the number of structures.
- The count is only printed: unlike `BOQ - Minas do Sol.ipynb`, it isn't stored in a DataFrame or exported to Excel.
- The comment "no C type on the edges" is a leftover from another notebook: there are no structure types here.

In [5]:
structure_df = structures.fetchdf()
structure_df['Line'] = round(structure_df['Y'],0)

structure_df.sort_values(by=["Line", "X"], ascending=[False,True], inplace=True, ignore_index=True)

#Different Lines
condition1 = structure_df["Line"] != structure_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = (abs(structure_df["X"] - structure_df["X"].shift()) > 13)


break_condition = condition1 | condition2

structure_df['Group'] = break_condition.cumsum()
print(f'The number of structures is: {max(structure_df['Group'])}')

The number of structures is: 187


CABLE DATA

The next cells measure the trenches, ducts and cables by layer.

Splitting the cable entities into groups
=
One relation per group, selected from `data` by layer name:
- `ac_cables`: `LIKE 'EMF_bt_vala_btca_%xC'` (AC trenches).
- `ac_access`: `LIKE 'EMF_bt_access_ductbank_%xC'`.
- `dc_cables`: exact match on the positive and negative solar cable layers.
- `dc_ducts`: `LIKE 'EMF_bt_eletroduto_%S'`.
- `dc_trenches`: exactly `EMF_DC_vala`.

Matching is case-sensitive. Layers that match nothing are ignored. There is no watercourse ductbank group in this project.

In [6]:
ac_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_vala_btca_%xC'
""")

ac_access = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_access_ductbank_%xC'
""")

dc_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_cabo_solar_negativo' OR layer = 'EMF_cabo_solar_positivo'
""")

dc_ducts = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_eletroduto_%S'
""")

dc_trenches = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_DC_vala'
""")

QTY. OF AC CABLES (m)
=

Prints the AC trench entities (`vala` = trench). The `NNxC` suffix of the layer is the number of AC circuits in the trench.

In [7]:
print(ac_cables)

┌───────────────────────┬───────────┬───────────┬───────────┬───────────┬────────────┐
│         layer         │  start_x  │  start_y  │   end_x   │   end_y   │   length   │
│        varchar        │   float   │   float   │   float   │   float   │   float    │
├───────────────────────┼───────────┼───────────┼───────────┼───────────┼────────────┤
│ EMF_bt_vala_btca_02xC │  692440.5 │  735624.6 │ 692439.25 │  735644.9 │  24.875591 │
│ EMF_bt_vala_btca_03xC │ 692439.25 │  735644.9 │  692435.5 │ 735654.25 │  10.123912 │
│ EMF_bt_vala_btca_06xC │  692435.5 │ 735654.25 │  692431.5 │ 735664.44 │ 10.9103155 │
│ EMF_bt_vala_btca_07xC │  692431.5 │ 735664.44 │  692427.6 │  735674.2 │  10.517114 │
│ EMF_bt_vala_btca_08xC │  692427.6 │  735674.2 │ 692405.94 │  735692.9 │  34.432583 │
│ EMF_bt_vala_btca_07xC │  692403.8 │ 735705.25 │  692403.8 │ 735710.25 │        5.0 │
│ EMF_bt_vala_btca_05xC │  692403.8 │ 735710.25 │ 692449.44 │ 735728.25 │    49.0691 │
│ EMF_bt_vala_btca_02xC │  692403.8 │ 73571

AC trench length per layer
=
`SUM(length)` per layer, sorted by layer name. The relation is referenced by its variable name (here written as a string). Goes to the `AC Trenches (m)` sheet.

In [8]:
ac_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_cables' GROUP BY layer ORDER BY layer")
print(ac_cables_group)

┌───────────────────────┬────────────────────┐
│         layer         │    total_length    │
│        varchar        │       double       │
├───────────────────────┼────────────────────┤
│ EMF_bt_vala_btca_01xC │  1097.795338153839 │
│ EMF_bt_vala_btca_02xC │ 485.59027576446533 │
│ EMF_bt_vala_btca_03xC │  619.8601217269897 │
│ EMF_bt_vala_btca_04xC │ 425.13274669647217 │
│ EMF_bt_vala_btca_05xC │ 231.53718185424805 │
│ EMF_bt_vala_btca_06xC │ 230.17827224731445 │
│ EMF_bt_vala_btca_07xC │ 181.24942874908447 │
│ EMF_bt_vala_btca_08xC │  472.0382595062256 │
│ EMF_bt_vala_btca_09xC │ 10.358613014221191 │
└───────────────────────┴────────────────────┘



QTY. OF AC DUCTS IN ACCESS (m and un.)
=

Prints the ductbanks where the AC trenches cross an access road.

In [9]:
print(ac_access)

┌─────────────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│            layer            │  start_x  │  start_y  │   end_x   │   end_y   │  length   │
│           varchar           │   float   │   float   │   float   │   float   │   float   │
├─────────────────────────────┼───────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_bt_access_ductbank_08xC │ 692333.25 │  735897.9 │ 692339.06 │ 735900.25 │  6.301596 │
│ EMF_bt_access_ductbank_06xC │  692597.2 │  736006.8 │ 692600.75 │  736002.0 │ 6.0000553 │
│ EMF_bt_access_ductbank_08xC │ 692779.44 │ 736141.56 │ 692783.25 │  736136.9 │  6.003517 │
│ EMF_bt_access_ductbank_08xC │ 693026.94 │  736118.7 │  693021.8 │  736121.8 │       6.0 │
└─────────────────────────────┴───────────┴───────────┴───────────┴───────────┴───────────┘



Access ductbank length per layer
=
`SUM(length)` per layer, sorted by layer name. The relation is referenced by its variable name (here written as a string). Goes to the `AC Ducts Access (m)` sheet.

In [10]:
ac_access_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_access' GROUP BY layer ORDER BY layer")
print(ac_access_group)

┌─────────────────────────────┬────────────────────┐
│            layer            │    total_length    │
│           varchar           │       double       │
├─────────────────────────────┼────────────────────┤
│ EMF_bt_access_ductbank_06xC │  6.000055313110352 │
│ EMF_bt_access_ductbank_08xC │ 18.305113315582275 │
└─────────────────────────────┴────────────────────┘



Access ductbanks: number of units
=
`COUNT(layer)` per layer, sorted by count. Each entity is one road crossing. Goes to the `AC Ducts Access (un.)` sheet.

In [11]:
ac_access_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'ac_access' GROUP BY layer ORDER BY count DESC")
print(ac_access_count)

┌─────────────────────────────┬───────┐
│            layer            │ count │
│           varchar           │ int64 │
├─────────────────────────────┼───────┤
│ EMF_bt_access_ductbank_08xC │     3 │
│ EMF_bt_access_ductbank_06xC │     1 │
└─────────────────────────────┴───────┘



QTY. OF DC CABLES (m)
=

Prints every positive and negative solar cable run. Several identical lines on top of each other (same start, end and length) are all summed, so each one must stand for a separate cable.

In [12]:
print(dc_cables)

┌─────────────────────────┬───────────┬──────────┬──────────┬───────────┬─────────┐
│          layer          │  start_x  │ start_y  │  end_x   │   end_y   │ length  │
│         varchar         │   float   │  float   │  float   │   float   │  float  │
├─────────────────────────┼───────────┼──────────┼──────────┼───────────┼─────────┤
│ EMF_cabo_solar_positivo │  692619.2 │ 736303.0 │ 692661.4 │ 736325.25 │ 66.8367 │
│ EMF_cabo_solar_positivo │  692619.2 │ 736303.0 │ 692661.4 │ 736325.25 │ 66.8367 │
│ EMF_cabo_solar_positivo │  692619.2 │ 736303.0 │ 692661.4 │ 736325.25 │ 66.8367 │
│ EMF_cabo_solar_positivo │  692654.3 │ 736303.0 │ 692661.4 │ 736325.25 │ 31.7117 │
│ EMF_cabo_solar_positivo │  692654.3 │ 736303.0 │ 692661.4 │ 736325.25 │ 31.7117 │
│ EMF_cabo_solar_positivo │ 692617.44 │ 736313.0 │ 692661.4 │ 736325.25 │ 59.7463 │
│ EMF_cabo_solar_positivo │ 692617.44 │ 736313.0 │ 692661.4 │ 736325.25 │ 59.7463 │
│ EMF_cabo_solar_positivo │ 692617.44 │ 736313.0 │ 692661.4 │ 736325.25 │ 59

DC cable length per layer and total
=
- Per-layer `SUM(length)` for the positive and the negative layers.
- The grand total (both polarities) is a second query, rounded to 3 decimals. Only the per-layer table is exported (`DC Cables (m)`).
- Overlapping identical lines are all counted: nothing removes duplicates, so every cable drawn on top of another one is summed again.

In [13]:
print(conn.sql('SELECT * FROM dc_cables ORDER BY length DESC LIMIT 1'))

┌─────────────────────────┬──────────┬──────────┬──────────┬───────────┬────────┐
│          layer          │ start_x  │ start_y  │  end_x   │   end_y   │ length │
│         varchar         │  float   │  float   │  float   │   float   │ float  │
├─────────────────────────┼──────────┼──────────┼──────────┼───────────┼────────┤
│ EMF_cabo_solar_positivo │ 692200.5 │ 735679.0 │ 692406.4 │ 735691.25 │ 251.75 │
└─────────────────────────┴──────────┴──────────┴──────────┴───────────┴────────┘



Longest DC cable
=
Sanity check only: prints the 1 longest DC cable run (`ORDER BY length DESC LIMIT 1`), to spot a cable that was drawn to the wrong inverter/combiner. Nothing is exported.

In [14]:
dc_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_cables' GROUP BY layer ORDER BY layer")
print(dc_cables_group)

print(f'The total length of DC cables is {round(conn.sql("SELECT SUM(length) FROM 'dc_cables'").fetchone()[0],3)} meters')

┌─────────────────────────┬────────────────────┐
│          layer          │    total_length    │
│         varchar         │       double       │
├─────────────────────────┼────────────────────┤
│ EMF_cabo_solar_negativo │ 145788.17309188843 │
│ EMF_cabo_solar_positivo │ 145788.17309188843 │
└─────────────────────────┴────────────────────┘

The total length of DC cables is 291576.346 meters


QTY. OF DC DUCTS (m and un.)
=

Prints the DC duct entities.

In [15]:
print(dc_ducts)

┌──────────────────────┬───────────┬──────────┬───────────┬───────────┬───────────┐
│        layer         │  start_x  │ start_y  │   end_x   │   end_y   │  length   │
│       varchar        │   float   │  float   │   float   │   float   │   float   │
├──────────────────────┼───────────┼──────────┼───────────┼───────────┼───────────┤
│ EMF_bt_eletroduto_1S │  692922.6 │ 736354.0 │ 692893.25 │  736289.0 │ 90.714966 │
│ EMF_bt_eletroduto_1S │  692627.4 │ 736332.0 │  692661.4 │  736325.5 │  36.77419 │
│ EMF_bt_eletroduto_1S │  692662.5 │ 736322.0 │  692661.4 │ 736325.25 │     4.375 │
│ EMF_bt_eletroduto_1S │  692983.7 │ 736290.0 │  692984.9 │  736283.6 │    7.5625 │
│ EMF_bt_eletroduto_1S │ 692892.06 │ 736286.0 │ 692893.25 │ 736288.75 │    3.9375 │
│ EMF_bt_eletroduto_1S │ 692892.06 │ 736276.0 │ 692893.25 │ 736288.75 │   13.9375 │
│ EMF_bt_eletroduto_1S │  692764.6 │ 736264.0 │  692762.6 │  736276.5 │   14.4773 │
│ EMF_bt_eletroduto_1S │ 692788.06 │ 736264.0 │ 692810.94 │ 736257.25 │ 25.0

DC duct length per layer
=
Per-layer sum for `1S` / `2S` / `3S` (the suffix is the number of conduits/strings in the duct). Goes to the `DC Ducts (m)` sheet.

In [16]:
dc_ducts_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_ducts' GROUP BY layer ORDER BY layer")
print(dc_ducts_group)

┌──────────────────────┬────────────────────┐
│        layer         │    total_length    │
│       varchar        │       double       │
├──────────────────────┼────────────────────┤
│ EMF_bt_eletroduto_1S │ 1823.8470516204834 │
│ EMF_bt_eletroduto_2S │  2073.819329738617 │
│ EMF_bt_eletroduto_3S │  6812.584581851959 │
└──────────────────────┴────────────────────┘



DC ducts: number of units
=
Number of duct entities per layer and in total. Each LINE/LWPOLYLINE on a duct layer counts as one duct. Goes to the `DC Ducts (un.)` sheet.

In [17]:
dc_ducts_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'dc_ducts' GROUP BY layer ORDER BY count DESC")
print(dc_ducts_count)

print(f'The total quantity of DC ducts is {conn.sql("SELECT COUNT(layer) FROM 'dc_ducts'").fetchone()[0]}')

┌──────────────────────┬───────┐
│        layer         │ count │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ EMF_bt_eletroduto_3S │   636 │
│ EMF_bt_eletroduto_2S │   101 │
│ EMF_bt_eletroduto_1S │    95 │
└──────────────────────┴───────┘

The total quantity of DC ducts is 832


QTY. OF DC TRENCHES (m)
=

Prints the DC trench entities.

In [18]:
print(dc_trenches)

┌─────────────┬───────────┬──────────┬───────────┬───────────┬─────────────┐
│    layer    │  start_x  │ start_y  │   end_x   │   end_y   │   length    │
│   varchar   │   float   │  float   │   float   │   float   │    float    │
├─────────────┼───────────┼──────────┼───────────┼───────────┼─────────────┤
│ EMF_DC_vala │  692406.4 │ 735688.0 │  692408.8 │  735688.0 │      2.4609 │
│ EMF_DC_vala │  692910.6 │ 736218.0 │  692922.4 │  736218.0 │       11.75 │
│ EMF_DC_vala │  692922.6 │ 736354.0 │  692922.6 │  736295.6 │     58.3447 │
│ EMF_DC_vala │  692903.2 │ 736295.6 │ 692901.56 │  736295.6 │   1.5798401 │
│ EMF_DC_vala │ 692603.94 │ 736342.0 │ 692603.94 │  736332.0 │        10.0 │
│ EMF_DC_vala │  692627.4 │ 736332.0 │  692661.4 │  736325.5 │    36.77419 │
│ EMF_DC_vala │  692627.4 │ 736332.0 │  692627.4 │  736332.1 │      0.1553 │
│ EMF_DC_vala │  692662.5 │ 736322.0 │  692661.4 │  736322.0 │       1.125 │
│ EMF_DC_vala │  692663.1 │ 736313.0 │  692661.4 │  736313.0 │      1.7794 │

DC trench length and total
=
Sum of `length` on `EMF_DC_vala` (one row, since there is a single trench layer), plus the grand total. Goes to the sheet named `DC Trenches (un.)`, although the value is a length in metres.

In [19]:
dc_trenches_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_trenches' GROUP BY layer ORDER BY layer")
print(dc_trenches_group)

print(f'\nThe total length of DC trenches is {round(conn.sql("SELECT SUM(length) FROM 'dc_trenches'").fetchone()[0],3)} meters')

┌─────────────┬────────────────────┐
│    layer    │    total_length    │
│   varchar   │       double       │
├─────────────┼────────────────────┤
│ EMF_DC_vala │ 2878.4053097963333 │
└─────────────┴────────────────────┘


The total length of DC trenches is 2878.405 meters


OVERVIEW
=

Writes every summary to its own sheet of `C:\Users\Usuario\Desktop\Toolestown I - BOQ.xlsx` (hardcoded path, overwritten on every run), then closes the DuckDB connection.
- The number of structures isn't exported.
- The `DC Trenches (un.)` sheet holds a length in metres, not units.

In [20]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\Toolestown I - BOQ.xlsx") as xlsx:
    ac_cables_group.df().to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    ac_access_group.df().to_excel(xlsx, sheet_name='AC Ducts Access (m)', index=False)
    ac_access_count.df().to_excel(xlsx, sheet_name='AC Ducts Access (un.)', index=False)
    dc_cables_group.df().to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group.df().to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_count.df().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group.df().to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)


conn.close()
